# ADP Technical Tets

In [31]:
!pip install -U pandas scikit-learn transformers[torch] datasets matplotlib seaborn accelerate 

zsh:1: no matches found: transformers[torch]


In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

topics_df = pd.read_csv('available_topics.csv')
conv_df = pd.read_csv('available_conversations.csv')

print("--- Available Topics ---")
display(topics_df[['topic_id', 'topic_name']])

--- Available Topics ---


,topic_id,topic_name
0,0,employee_benefits
1,1,employee_training
2,2,other
3,3,payroll
4,4,performance_management
5,5,talent_acquisition
6,6,tax_services
7,7,time_and_attendance


In [ ]:
# Split into 80% training and 20% test set to evaluate the model
train_texts, test_texts, train_labels, test_labels = train_test_split(
    conv_df['message'], 
    conv_df['topic_id'], 
    test_size=0.2, 
    random_state=42,
    stratify=conv_df['topic_id']
)

print(f"Training set size: {len(train_texts)}")
print(f"Test set size: {len(test_texts)}")

Training set size: 1920
Test set size: 480


In [34]:
print("--- Training Baseline Model ---")

# Convert text to numerical features
vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)
X_train_tfidf = vectorizer.fit_transform(train_texts)
X_test_tfidf = vectorizer.transform(test_texts)

# Train model (generates a single label per message)
baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train_tfidf, train_labels)
baseline_preds = baseline_model.predict(X_test_tfidf)

print("\nClassification Report (Baseline):")
print(classification_report(test_labels, baseline_preds, target_names=topics_df['topic_name'].values))

--- Training Baseline Model ---

Classification Report (Baseline):
                        precision    recall  f1-score   support

     employee_benefits       0.55      0.57      0.56        60
     employee_training       0.39      0.40      0.40        60
                 other       0.42      0.37      0.39        60
               payroll       0.38      0.48      0.42        60
performance_management       0.34      0.43      0.38        60
    talent_acquisition       0.49      0.37      0.42        60
          tax_services       0.48      0.48      0.48        60
   time_and_attendance       0.52      0.40      0.45        60

              accuracy                           0.44       480
             macro avg       0.45      0.44      0.44       480
          weighted avg       0.45      0.44      0.44       480

